# Task 1.1 — Core Contribution / Architecture
**Paper**: *Efficient Online Learning for Large-Scale Sparse Kernel Logistic Regression*  
**Authors**: Lijun Zhang, Rong Jin, Chun Chen, Jiajun Bu, Xiaofei He  
**Venue**: AAAI 2012

## Step-by-Step Method Description

The paper tackles the problem of training Kernel Logistic Regression (KLR) classifiers on large datasets. Standard KLR produces dense kernel classifiers because the logit loss function never exactly zeros out, so every training point ends up contributing to the final model. The authors propose *conservative* online learning algorithms that stochastically skip updates for well-classified examples, yielding sparse classifiers without sacrificing the theoretical convergence guarantee.

Below I walk through the method from input to output.

### Step 1: Problem Formulation — Constrained Kernel Logistic Regression

- **Description**: The authors formulate KLR as a constrained optimisation problem over a Reproducing Kernel Hilbert Space (RKHS) $\mathcal{H}_\kappa$. Instead of the regularised form $\min_{f}\frac{\lambda}{2}\|f\|^2 + \frac{1}{n}\sum \ell(y_i f(x_i))$, they use a norm-constrained version: minimise the average logit loss $\frac{1}{n}\sum_{i=1}^{n} \ell(y_i f(x_i))$ subject to $\|f\|_{\mathcal{H}_\kappa} \le R$, where $\ell(z) = \ln(1 + e^{-z})$ is the logistic loss.
- **Reference**: Equation (2) in the paper.
- **Purpose**: Constraining the function norm to a ball of radius $R$ makes the online projection step straightforward (a simple rescaling) and decouples regularisation strength from the step-size choice.

### Step 2: Non-Conservative Online Update (NC — Algorithm 1)

- **Description**: Starting from $f_1 = 0$, the algorithm processes one training example $(x_t, y_t)$ at a time. It computes the posterior probability $p(y_t | f_t(x_t)) = \sigma(y_t f_t(x_t))$ and then performs a functional gradient descent step: $f'_{t+1}(x) = f_t(x) - \eta\, y_t\,[p(y_t|f_t(x_t)) - 1]\,\kappa(x_t, x)$. After every single example, a new support vector $x_t$ is added to the classifier.
- **Reference**: Equation (3)–(4) and Algorithm 1, lines 5–6.
- **Purpose**: This is the baseline online learner. It establishes the convergence guarantee (Theorem 1: $O(R/\sqrt{T})$ excess risk) but produces a completely dense classifier — every training point becomes a support vector. The convergence behaviour of NC is visualised in Figure 1 of the paper, where accuracy improves with more training examples but the number of support vectors equals $T$.

### Step 3: Projection onto the Constraint Set $\Omega$

- **Description**: After each gradient update, the updated function $f'_{t+1}$ may violate the norm constraint. The algorithm projects it back: $f_{t+1} = \pi_\Omega(f'_{t+1}) = \frac{R}{\max(R, \|f'_{t+1}\|_{\mathcal{H}_\kappa})} \cdot f'_{t+1}$. In practice this rescales all the support-vector weights by the same factor.
- **Reference**: Algorithm 1, line 7; definition of $\pi_\Omega$ in the "Non-conservative" section.
- **Purpose**: Keeps the classifier inside the feasible set $\Omega$, which is required for the convergence proofs (Theorems 1–4) and prevents the function norm from blowing up during training.

### Step 4: Classification-Margin-Based Conservative Update (Margin — Algorithm 2)

- **Description**: Before performing the gradient update, the algorithm draws a Bernoulli random variable $Z_t \sim \text{Bernoulli}(p_t)$ with sampling probability $p_t = \frac{2-\eta}{2-\eta + \eta\, p(y_t|f_t(x_t))}$ (Eq. 5). If $Z_t = 1$ the update proceeds exactly as in NC; if $Z_t = 0$ the classifier is left unchanged ($f_{t+1} = f_t$). Because $p_t$ decreases as the posterior confidence $p(y_t|f_t(x_t))$ increases, well-classified points are more likely to be skipped.
- **Reference**: Equation (5) and Algorithm 2.
- **Purpose**: This is the first core contribution. By skipping updates stochastically, fewer training examples become support vectors, producing a *sparser* classifier. Theorem 3 shows the generalisation bound remains identical to Theorem 2 for NC.

### Step 5: Auxiliary-Function-Based Conservative Update (Auxiliary — Algorithm 3)

- **Description**: A drawback of the Margin algorithm is that when $\eta$ is small, $p_t \approx 1$ and almost no updates are skipped. The Auxiliary algorithm decouples the sampling probability from $\eta$ by introducing an auxiliary function $h(z) \ge \ell(z)$ that is convex. The sampling probability becomes $p_t = \ell(y_t f_t(x_t)) / h(y_t f_t(x_t))$ (Eq. 6). When $Z_t = 1$, the gradient of $h$ (not $\ell$) is used for the update. The paper studies three choices, the main one being $h(z) = \ln(\gamma + e^{-z})$ with $\gamma \ge 1$. Increasing $\gamma$ pushes $p_t$ toward 0 for correctly classified points, dramatically improving sparsity.
- **Reference**: Equation (6), Algorithm 3, and the three auxiliary-function variants listed after Theorem 4.
- **Purpose**: This is the paper's main algorithmic contribution. It gives the user direct control over the sparsity–accuracy trade-off via $\gamma$, while Theorem 4 still provides a convergence guarantee. Figure 4 of the paper shows how sparsity varies with $\gamma - 1$: increasing $\gamma$ monotonically increases sparsity while accuracy degrades gracefully until $\gamma$ becomes extreme.

### Step 6: Output — Averaged Classifier

- **Description**: After processing all $T$ examples, the final output is the average over all intermediate classifiers: $\hat{f}(x) = \frac{1}{T}\sum_{t=1}^{T} f_t(x)$.
- **Reference**: Algorithm 1 line 9, Algorithm 2 line 15, Algorithm 3 line 15.
- **Purpose**: Averaging is a standard technique in online-to-batch conversion; it smooths out the noise from individual stochastic gradient steps and is essential for the $O(1/\sqrt{T})$ convergence rate.

## Final Summary Sentence

This paper solves the problem of obtaining *sparse* kernel classifiers for large-scale logistic regression — something that standard KLR optimisers and the Import Vector Machine (IVM) could not do efficiently — by introducing stochastic conservative update rules (Bernoulli gating) that skip gradient updates for confidently classified examples, and the authors show that these conservative algorithms retain the same $O(1/\sqrt{T})$ generalisation bound as the dense non-conservative baseline while using substantially fewer support vectors.